In [2]:
!pip install cvxpy control --quiet
print('OK')

OK



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
!pip install casadi


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
"""
Koopman Closed-Loop v4b-fix — SIMULACION (sincronizada con planta real)
=========================================================================
Esta version de SIMULACION reemplaza el ESP32Interface por el modelo RK4
con ruido gaussiano (sigma=1mm en h1,h3, replica HC-SR04), pero mantiene
TODO lo demas IDENTICO al script que corre sobre la planta fisica real
v4b-fix:

  - KI_WEIGHT       = 80.0    (antes 50.0 en "v4" simulacion original)
  - Z_INT_LIMIT     = 0.015   (antes 0.03)
  - Peso terminal DISTRIBUIDO: wk=5 en los ultimos N_TERMINAL=3 pasos,
    wk=1 el resto (antes wk=10 SOLO en el ultimo paso)
  - Anti-windup con FREEZE del integrador durante TRANSIENT_STEPS=30
    pasos (~60s) tras cada cambio de referencia, ADEMAS del deadband
    de 3cm (antes solo deadband, sin freeze por tiempo)
  - Rampa de referencia interna (T_RAMP_PER_10CM=40s) que ve el MPC
    internamente via yr_ramp (la referencia "cruda" yr NO se le pasa
    directamente al solver durante el transitorio)
  - Warm-start con z_int=0 EXPLICITO al cambiar de referencia, usando
    los puntos de equilibrio SS_NOM (antes w0_mpc=None generico)

DIAGNOSTICO INCLUIDO:
  Se agrega el flag ENABLE_REF_RAMP (linea ~45). Ponerlo en False
  desactiva la rampa (el MPC ve el escalon crudo, igual que el NMPC de
  referencia) sin tocar nada mas. Esto permite verificar en minutos,
  SOLO EN SIMULACION, si el "tiempo de asentamiento" mas lento de
  Koopman en el segmento +10cm es un efecto de la rampa o un efecto
  genuino del observador/controlador. Compara T_s^5% del Seg.2 con
  ENABLE_REF_RAMP=True vs False.

Todo lo demas (parametros fisicos, EDMD, observador, diccionario de 8
observables, N_mpc=20, q=500, R1, R2) es identico a la simulacion v4
original y a la planta real.
"""

import numpy as np
from scipy.optimize import least_squares
import scipy.linalg as la
import casadi as ca
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import time
import csv
from datetime import datetime

warnings.filterwarnings('ignore')
output_dir = Path('./outputs_sim')
output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# PARAMETROS FISICOS - idénticos a v4b-fix real
# ============================================================================
P_NOM = {
    'Atank'   : 3.02e-3,
    'rho'     : 997.0,
    'eta'     : 8.9e-4,
    'g'       : 9.81,
    'alphao1' : 0.0271, 'Do1': 10e-3,
    'alphao2' : 0.0271, 'A2' : 7.85e-5,
    'alphao3' : 0.0271, 'Do3': 10e-3,
    'alpha120': 0.3038, 'D12': 10e-3, 'A12': 7.85e-5, 'lambdac12': 24000,
    'alpha230': 0.1344, 'D23': 10e-3, 'A23': 7.85e-5, 'lambdac23': 29600,
    'qi1max'  : 2.69e-5,
    'qi3max'  : 2.41e-5,
}

Ts         = 2.0
T_sim      = 800
Nsim       = int(T_sim / Ts)
sigma      = 1e-3        # ruido HC-SR04 simulado (1 mm)
EPS        = 1e-10
EPS_PSI    = 1e-8
SETTLING_S = 60

REF_STEPS = [
    (0,              0.10),
    (int(Nsim*0.33), 0.20),
    (int(Nsim*0.66), 0.15),
]

# ============================================================================
# PARAMETROS DEL OBSERVADOR E INTEGRADOR - IDENTICOS A v4b-fix REAL
# ============================================================================
Q_OBSERVER      = 0.1
KI_WEIGHT       = 80.0
DEADBAND_H2     = 0.03
Z_INT_LIMIT     = 0.015
T_RAMP_PER_10CM = 40.0
TRANSIENT_STEPS = 30

# --- TOGGLE DE DIAGNOSTICO -------------------------------------------------
# False => MPC ve el escalon crudo (igual que el NMPC de referencia).
# Util para verificar si la rampa explica el Ts_seg2 reportado.
ENABLE_REF_RAMP = True
# ----------------------------------------------------------------------------

# ============================================================================
# MODELO INTERNO (idéntico a v4 / v4b-fix)
# ============================================================================
def nl_ode(x, u, p=P_NOM):
    h1 = max(x[0], EPS); h2 = max(x[1], EPS); h3 = max(x[2], EPS)
    qi1 = p['qi1max'] * np.clip(u[0], 0, 1)
    qi3 = p['qi3max'] * np.clip(u[1], 0, 1)
    qo1 = p['alphao1'] * (p['Do1']**2 * np.pi / 4) * np.sqrt(2 * p['g'] * h1)
    qo2 = p['alphao2'] * p['A2']                    * np.sqrt(2 * p['g'] * h2)
    qo3 = p['alphao3'] * (p['Do3']**2 * np.pi / 4) * np.sqrt(2 * p['g'] * h3)
    dh12 = h1 - h2
    l12  = p['D12'] * p['rho'] / p['eta'] * np.sqrt(2 * p['g'] * abs(dh12) + EPS)
    a12  = p['alpha120'] * np.tanh(2 * l12 / p['lambdac12'])
    q12  = a12 * p['A12'] * np.sqrt(2 * p['g'] * abs(dh12) + EPS) * np.sign(dh12 + 1e-15)
    dh23 = h2 - h3
    l23  = p['D23'] * p['rho'] / p['eta'] * np.sqrt(2 * p['g'] * abs(dh23) + EPS)
    a23  = p['alpha230'] * np.tanh(2 * l23 / p['lambdac23'])
    q23  = a23 * p['A23'] * np.sqrt(2 * p['g'] * abs(dh23) + EPS) * np.sign(dh23 + 1e-15)
    return np.array([
        (qi1 - q12 - qo1) / p['Atank'],
        (q12 - q23 - qo2) / p['Atank'],
        (qi3 + q23 - qo3) / p['Atank'],
    ])

def rk4(x, u, p=P_NOM, dt=Ts):
    k1 = nl_ode(x, u, p); k2 = nl_ode(x + dt/2*k1, u, p)
    k3 = nl_ode(x + dt/2*k2, u, p); k4 = nl_ode(x + dt*k3, u, p)
    return np.clip(x + dt/6*(k1 + 2*k2 + 2*k3 + k4), 0, 0.27)

def psi(x):
    h1, h2, h3 = x[0], x[1], x[2]
    return np.array([
        np.sqrt(max(h1, EPS_PSI)),
        np.sqrt(max(h2, EPS_PSI)),
        np.sqrt(max(h3, EPS_PSI)),
        h1 * h2, h2 * h3, h1 * h3,
        h1, h3,
    ], dtype=float)

def compute_ss(h2_ref, p=P_NOM):
    w_reg = 0.05
    def eqs(v):
        h1, h3, u1, u3 = v
        f = nl_ode([max(h1, 1e-4), h2_ref, max(h3, 1e-4)],
                   [np.clip(u1, 0, 1), np.clip(u3, 0, 1)], p)
        return np.concatenate([f, [w_reg * u3]])
    guesses = [
        (h2_ref*1.4, h2_ref*0.8, 0.5, 0.0), (h2_ref*1.6, h2_ref*0.7, 0.7, 0.0),
        (h2_ref*2.0, h2_ref*0.8, 0.9, 0.0), (h2_ref*2.5, h2_ref*0.8, 0.7, 0.0),
        (h2_ref*2.5, h2_ref*0.8, 0.75, 0.0),(h2_ref*1.8, h2_ref*0.85, 0.85, 0.0),
        (h2_ref*1.3, h2_ref*0.9, 1.0, 0.1), (h2_ref*1.2, h2_ref*1.0, 1.0, 0.3),
        (h2_ref*1.1, h2_ref*1.1, 1.0, 0.5), (h2_ref*1.0, h2_ref*1.2, 1.0, 0.8),
    ]
    best, best_res = None, 1e10
    for g in guesses:
        try:
            s = least_squares(eqs, list(g),
                              bounds=([0.01, 0.01, 0.0, 0.0], [0.27, 0.27, 1.0, 1.0]),
                              max_nfev=5000)
            phys_res = np.linalg.norm(s.fun[:3])
            if s.success and phys_res < best_res:
                best_res = phys_res; best = s.x
        except: pass
    if best is not None and best_res < 1e-4:
        return (np.array([best[0], h2_ref, best[1]]),
                np.array([np.clip(best[2], 0, 1), np.clip(best[3], 0, 1)]))
    return None, None

# ============================================================================
# ENTRENAMIENTO EDMD - idéntico a v4 / v4b-fix
# ============================================================================
def train_edmd(verbose=True):
    if verbose:
        print("\nEntrenando EDMD (parametros planta real, v4b-fix)...")
    rng = np.random.default_rng(42)
    z_k, u_k, z_k1 = [], [], []

    for h2r in [0.10, 0.12, 0.15, 0.18, 0.20, 0.22]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is None: continue
        for pert in [0.95, 0.90, 0.98, 0.85, 0.92, 0.88]:
            x = x_ss.copy(); x[1] = np.clip(h2r * pert, 0.05, 0.70)
            for _ in range(170):
                err = x[1] - h2r; u = u_ss.copy()
                u[0] += np.clip(-0.35*err, -0.35, 0.35) + rng.uniform(-0.15, 0.15)
                u[1] += rng.uniform(-0.05, 0.05); u = np.clip(u, 0, 1)
                z = psi(x); xn = rk4(x, u); zn = psi(xn)
                z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    for h2r in [0.10, 0.15, 0.20]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is None: continue
        for pert in [0.60, 0.70, 0.75, 1.25, 1.35]:
            x = x_ss.copy(); x[1] = np.clip(h2r * pert, 0.05, 0.70)
            for _ in range(100):
                err = x[1] - h2r; u = u_ss.copy()
                u[0] += np.clip(-0.6*err, -0.6, 0.6) + rng.uniform(-0.1, 0.1)
                u = np.clip(u, 0, 1)
                z = psi(x); xn = rk4(x, u); zn = psi(xn)
                z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    for (h2f, h2t) in [(0.10, 0.20), (0.20, 0.15), (0.15, 0.10)]:
        xf, uf = compute_ss(h2f); xt, _ = compute_ss(h2t)
        if xf is None or xt is None: continue
        x = xf.copy()
        for _ in range(200):
            err = x[1] - h2t; u = uf.copy()
            u[0] += np.clip(-0.5*err, -0.5, 0.5) + rng.uniform(-0.08, 0.08)
            u = np.clip(u, 0, 1)
            z = psi(x); xn = rk4(x, u); zn = psi(xn)
            z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    Z = np.array(z_k); U = np.array(u_k); Zn = np.array(z_k1)
    nz, nu = Z.shape[1], U.shape[1]
    Phi = np.hstack([Z, U]); lam = 1e-4
    Theta = np.linalg.solve(Phi.T @ Phi + lam * np.eye(nz + nu), Phi.T @ Zn)
    A_raw = Theta[:nz, :].T; B_d = Theta[nz:, :].T
    vals, vecs = np.linalg.eig(A_raw); delta = 1e-4
    vals_s = np.where(np.abs(vals) >= 1.0, vals / np.abs(vals) * (1.0 - delta), vals)
    A_d = np.real(vecs @ np.diag(vals_s) @ np.linalg.inv(vecs))
    rmse = np.sqrt(np.mean((Z @ A_d.T + U @ B_d.T - Zn)**2))
    eigs = np.abs(np.linalg.eigvals(A_d))
    if verbose:
        print(f"  {len(Z)} muestras de entrenamiento")
        print(f"  RMSE one-step: {rmse:.2e}")
        print(f"  |lambda| in [{eigs.min():.5f}, {eigs.max():.5f}]")
    return A_d, B_d, nz, nu

# ============================================================================
# DISEÑO DEL OBSERVADOR - idéntico a v4 / v4b-fix
# ============================================================================
def design_observer(A_d, nz):
    C_obs = np.zeros((4, nz))
    C_obs[0, 0] = 1.0; C_obs[1, 2] = 1.0
    C_obs[2, 6] = 1.0; C_obs[3, 7] = 1.0
    m = C_obs.shape[0]
    print(f"\nDisenando ganancia L (DARE con Q={Q_OBSERVER}*I)...")
    for q_try in [Q_OBSERVER, 0.05, 0.5, 1.0, 5.0, 10.0]:
        try:
            Q_obs = q_try * np.eye(nz)
            X = la.solve_discrete_are(A_d.T, C_obs.T, Q_obs, np.eye(m))
            L_d = np.linalg.solve(C_obs @ X @ C_obs.T + np.eye(m), C_obs @ X @ A_d.T).T
            max_eig = np.abs(np.linalg.eigvals(A_d - L_d @ C_obs)).max()
            if max_eig < 1.0 and np.abs(L_d).max() > 0.01:
                print(f"  q={q_try}: max|lambda(A-LC)|={max_eig:.5f}, max|L|={np.abs(L_d).max():.4f}")
                print(f"  tau_decay = {-Ts/np.log(max_eig):.1f} s")
                return L_d, C_obs
        except: pass
    raise RuntimeError("No se pudo disenar L.")

# ============================================================================
# CONSTRUCCION DEL MPC CON INTEGRADOR (v4b-fix: peso terminal distribuido)
# ============================================================================
def build_mpc_with_integrator(A_d, B_d, nz, nu):
    print(f"\nConstruyendo KoopmanMPC v4b-fix-sim con integrador (ki={KI_WEIGHT})...")
    N_mpc = 20; q_mpc = 500
    R1 = np.diag([0.1, 1.0]); R2 = np.diag([0.1, 0.1])
    umin = np.array([0.0, 0.0]); umax = np.array([1.0, 1.0])
    nz_aug = nz + 1

    N_TERMINAL = 3
    W_TERMINAL = 5

    Z_sym  = ca.SX.sym('Z',  nz_aug, N_mpc+1)
    U_sym  = ca.SX.sym('U',  nu,     N_mpc)
    z0_sym = ca.SX.sym('z0', nz_aug)
    yr_sym = ca.SX.sym('yr')
    up_sym = ca.SX.sym('up', nu)
    A_cas  = ca.DM(A_d); B_cas = ca.DM(B_d)

    J = 0; g = []; lbg = []; ubg = []
    g += [Z_sym[:, 0] - z0_sym]; lbg += [0.0]*nz_aug; ubg += [0.0]*nz_aug

    for k in range(N_mpc):
        zk    = Z_sym[:nz, k];   zk1   = Z_sym[:nz, k+1]
        zint  = Z_sym[nz, k];    zint1 = Z_sym[nz, k+1]
        g    += [zk1 - (A_cas @ zk + B_cas @ U_sym[:, k])]
        lbg  += [0.0]*nz; ubg += [0.0]*nz
        h2_est_k = zk[1]**2
        g    += [zint1 - (zint + (yr_sym - h2_est_k) * Ts)]
        lbg  += [0.0]; ubg += [0.0]
        h2_k  = zk1[1]**2
        ey    = h2_k - yr_sym
        du    = U_sym[:, 0] - up_sym if k == 0 else U_sym[:, k] - U_sym[:, k-1]

        wk = W_TERMINAL if k >= (N_mpc - N_TERMINAL) else 1

        J += (wk * q_mpc * ey**2
              + KI_WEIGHT * zint**2
              + ca.mtimes(U_sym[:, k].T, ca.DM(R1) @ U_sym[:, k])
              + ca.mtimes(du.T,          ca.DM(R2) @ du))

    w_sym   = ca.vertcat(ca.reshape(Z_sym, nz_aug*(N_mpc+1), 1),
                         ca.reshape(U_sym, nu*N_mpc, 1))
    par_sym = ca.vertcat(z0_sym, yr_sym, up_sym)
    lbw = [-ca.inf]*(nz_aug*(N_mpc+1)) + list(np.tile(umin, N_mpc))
    ubw = [ ca.inf]*(nz_aug*(N_mpc+1)) + list(np.tile(umax, N_mpc))
    nlp  = {'x': w_sym, 'f': J, 'g': ca.vertcat(*g), 'p': par_sym}
    opts = {'ipopt.print_level': 0, 'ipopt.max_iter': 500,
            'ipopt.tol': 1e-5, 'ipopt.acceptable_tol': 1e-4, 'print_time': 0}
    ki_str = str(KI_WEIGHT).replace('.', 'p')
    solver = ca.nlpsol(f'kmpc_v4bfix_sim_{ki_str}', 'ipopt', nlp, opts)
    print(f"  KoopmanMPC v4b-fix-sim listo (nz_aug={nz_aug})")
    print(f"  Peso terminal: wk={W_TERMINAL} en ultimos {N_TERMINAL} pasos")
    return solver, N_mpc, umin, umax, lbw, ubw, lbg, ubg, nz_aug, nu

# ============================================================================
# LOOP DE SIMULACION - replica v4b-fix sin ESP32
# ============================================================================
def run_simulation(A_d, B_d, L_d, C_obs,
                    solver, N_mpc, umin, umax, lbw, ubw, lbg, ubg,
                    nz_aug, nu, nz, SS_NOM):

    rng = np.random.default_rng(42)

    yref_v = np.zeros(Nsim)
    for (k_start, ref_val) in REF_STEPS:
        yref_v[k_start:] = ref_val
    ref_change_times_s = [k * Ts for k, _ in REF_STEPS[1:]]

    x_ss0, u_ss0 = SS_NOM[0.10]
    x_true = x_ss0.copy()

    z_koopman = psi(x_true)
    z_int     = 0.0
    z_aug     = np.append(z_koopman, z_int)
    uprev     = np.array([0.0, 0.0])
    w0_mpc    = None
    yr_prev   = yref_v[0]

    in_transient = True
    transient_counter = 0

    yr_ramp_start = x_true[1]
    yr_target     = yref_v[0]
    yr_ramp       = yr_ramp_start
    ramp_active   = ENABLE_REF_RAMP

    T_hist  = np.arange(Nsim + 1) * Ts
    X_true  = np.zeros((3, Nsim + 1)); X_true[:, 0] = x_true
    X_hat   = np.zeros((3, Nsim + 1)); X_hat[:, 0]  = x_true
    Z_INT   = np.zeros(Nsim + 1);      Z_INT[0]     = z_int
    U_hist  = np.zeros((2, Nsim))
    step_ms = []

    print(f"\n{'='*65}")
    print(f"  SIMULACION Koopman CL v4b-fix | 0.10->0.20->0.15m | sigma={sigma*1000:.0f}mm")
    print(f"  KI_WEIGHT={KI_WEIGHT} | Z_INT_LIMIT=+-{Z_INT_LIMIT*100:.1f}cm.s | "
          f"DEADBAND={DEADBAND_H2*100:.0f}cm | RAMP={'ON' if ENABLE_REF_RAMP else 'OFF'}")
    print(f"{'='*65}")

    t_sim_start = time.time()

    for k in range(Nsim):
        t_step = time.time()
        yr = yref_v[k]

        if yr != yr_prev:
            z_int = 0.0; Z_INT[k] = 0.0
            in_transient = True; transient_counter = 0

            yr_ramp_start = X_hat[1, k] if k > 0 else x_true[1]
            yr_target     = yr
            yr_ramp       = yr_ramp_start
            ramp_active   = ENABLE_REF_RAMP

            if yr in SS_NOM:
                z_ss_new = psi(SS_NOM[yr][0]); u_ss_new = SS_NOM[yr][1]
                z_aug_ss = np.append(z_ss_new, 0.0)
                Z0 = np.tile(z_aug_ss, (N_mpc+1, 1)); U0 = np.tile(u_ss_new, (N_mpc, 1))
                w0_mpc = np.concatenate([Z0.flatten(), U0.flatten()])
            else:
                w0_mpc = None
            print(f"\n  -> Cambio referencia: {yr_prev*100:.0f}->{yr*100:.0f} cm | Reset z_int")
        yr_prev = yr

        if ramp_active:
            step_size = abs(yr_target - yr_ramp_start)
            if step_size > 1e-4:
                ramp_rate  = 0.10 / T_RAMP_PER_10CM   # m/s fijo
                delta_ramp = ramp_rate * Ts
                if yr_target > yr_ramp_start:
                    yr_ramp = min(yr_ramp + delta_ramp, yr_target)
                else:
                    yr_ramp = max(yr_ramp - delta_ramp, yr_target)
            if abs(yr_ramp - yr_target) < 1e-4:
                yr_ramp = yr_target
                ramp_active = False
        else:
            yr_ramp = yr_target

        transient_counter += 1
        if in_transient and transient_counter > TRANSIENT_STEPS:
            in_transient = False

        # 1. Prediccion Koopman
        z_koopman_pred = A_d @ z_aug[:nz] + B_d @ uprev
        z_koopman_pred[:3] = np.maximum(z_koopman_pred[:3], 0)
        z_koopman_pred[6:] = np.maximum(z_koopman_pred[6:], 0)

        # 2. Planta simulada
        x_new = rk4(x_true, uprev)
        X_true[:, k+1] = x_new
        x_true = x_new

        # 3. Medicion con ruido (h1, h3) - replica HC-SR04
        h1_meas = max(x_true[0] + sigma * rng.standard_normal(), 0.0)
        h3_meas = max(x_true[2] + sigma * rng.standard_normal(), 0.0)

        # 4. Correccion Luenberger
        y_obs = np.array([
            np.sqrt(max(h1_meas, EPS_PSI)),
            np.sqrt(max(h3_meas, EPS_PSI)),
            h1_meas, h3_meas,
        ])
        innov     = y_obs - C_obs @ z_koopman_pred
        z_koopman = z_koopman_pred + L_d @ innov
        z_koopman[:3] = np.maximum(z_koopman[:3], 0)
        z_koopman[6:] = np.maximum(z_koopman[6:], 0)

        # 5. Estado fisico estimado
        x_hat  = np.clip(z_koopman[:3]**2, 0, 0.75)
        h2_est = x_hat[1]
        X_hat[:, k+1] = x_hat

        # 6. Integrador con freeze en transitorio (idéntico a v4b-fix)
        error_h2 = yr_ramp - h2_est
        if not in_transient and not ramp_active:
            if abs(error_h2) < DEADBAND_H2:
                z_int += error_h2 * Ts
                z_int  = np.clip(z_int, -Z_INT_LIMIT, Z_INT_LIMIT)
        Z_INT[k+1] = z_int

        # 7. Estado aumentado
        z_aug = np.append(z_koopman, z_int)

        # 8. KoopmanMPC (usa yr_ramp como referencia interna)
        if w0_mpc is None:
            w0_mpc = np.zeros(nz_aug*(N_mpc+1) + nu*N_mpc)
            w0_mpc[:nz_aug] = z_aug

        par_val = np.concatenate([z_aug, [yr_ramp], uprev])
        try:
            sol    = solver(x0=w0_mpc, lbx=lbw, ubx=ubw,
                            lbg=lbg, ubg=ubg, p=par_val)
            w_opt  = np.array(sol['x']).flatten()
            Z_opt  = w_opt[:nz_aug*(N_mpc+1)].reshape(N_mpc+1, nz_aug)
            U_opt  = w_opt[nz_aug*(N_mpc+1):].reshape(N_mpc, nu)
            u_k    = np.clip(U_opt[0], umin, umax)
            Z_sh   = np.vstack([Z_opt[1:], Z_opt[-1:]])
            U_sh   = np.vstack([U_opt[1:], U_opt[-1:]])
            w0_mpc = np.concatenate([Z_sh.flatten(), U_sh.flatten()])
        except Exception as e:
            print(f"  MPC fallo en k={k}: {e}")
            u_k = uprev.copy(); w0_mpc = None

        elapsed_ms = (time.time() - t_step) * 1000
        step_ms.append(elapsed_ms)
        U_hist[:, k] = u_k; uprev = u_k.copy()

        if k % 50 == 0:
            print(f"  {k*Ts:>6.0f} | h2_real={x_true[1]*100:>7.3f} | "
                  f"h2_est={h2_est*100:>7.3f} | ref={yr*100:>6.1f} | "
                  f"u1={u_k[0]:>5.3f} | {elapsed_ms:>5.0f}ms")

    print(f"\n  Completado en {time.time()-t_sim_start:.1f}s")

    return {
        'T_hist': T_hist, 'X_true': X_true, 'X_hat': X_hat,
        'Z_INT': Z_INT, 'U_hist': U_hist, 'yref_v': yref_v,
        'step_ms': step_ms, 'ref_change_times_s': ref_change_times_s,
    }

# ============================================================================
# METRICAS (mismo esquema que v4b-fix real: ctrl, obs, h1, h3)
# ============================================================================
def compute_metrics(res):
    T   = res['T_hist']
    ref = np.append(res['yref_v'], res['yref_v'][-1])
    rct = res['ref_change_times_s']

    idx_400 = T > 400
    settling_mask = np.ones(len(T), dtype=bool)
    for tc in rct:
        settling_mask &= ~((T >= tc) & (T < tc + SETTLING_S))
    mask_ss = idx_400 & settling_mask

    e_h2_ctrl = (res['X_hat'][1, :] - ref) * 100
    e_obs     = (res['X_hat'][1, :] - res['X_true'][1, :]) * 100
    e_h1_est  = (res['X_hat'][0, :] - res['X_true'][0, :]) * 100
    e_h3_est  = (res['X_hat'][2, :] - res['X_true'][2, :]) * 100

    rmse_ctrl   = float(np.sqrt(np.mean(e_h2_ctrl[mask_ss]**2)))
    bias_ctrl   = float(np.mean(e_h2_ctrl[mask_ss]))
    mae_ctrl    = float(np.mean(np.abs(e_h2_ctrl[mask_ss])))
    rmse_trans  = float(np.sqrt(np.mean(e_h2_ctrl[idx_400]**2)))
    rmse_obs_ss = float(np.sqrt(np.mean(e_obs[mask_ss]**2)))
    bias_obs_ss = float(np.mean(e_obs[mask_ss]))
    rmse_h1     = float(np.sqrt(np.mean(e_h1_est[mask_ss]**2)))
    rmse_h3     = float(np.sqrt(np.mean(e_h3_est[mask_ss]**2)))

    # T_s 5% y overshoot por segmento (mismo criterio que tabla `transient`)
    seg_info = []
    bounds = [0] + rct + [T[-1]]
    refs_seg = [res['yref_v'][0]]
    for tc in rct:
        idx = np.searchsorted(T, tc)
        refs_seg.append(res['yref_v'][min(idx, len(res['yref_v'])-1)])
    for i in range(len(bounds)-1):
        t0, t1 = bounds[i], bounds[i+1]
        r = refs_seg[i]
        m = (T >= t0) & (T < t1)
        h2_seg = res['X_hat'][1, m]
        band = 0.05 * (abs(r - refs_seg[i-1]) if i > 0 else r)
        idx_in = np.where(np.abs(h2_seg - r) <= max(band, 1e-6))[0]
        ts5 = (T[m][idx_in[0]] - t0) if len(idx_in) else np.nan
        os_seg = float(np.max(h2_seg - r)) * 100 if i > 0 else np.nan
        seg_info.append((r, ts5, os_seg))

    return {
        'rmse_ctrl': rmse_ctrl, 'bias_ctrl': bias_ctrl, 'mae_ctrl': mae_ctrl,
        'rmse_trans': rmse_trans, 'rmse_obs_ss': rmse_obs_ss,
        'bias_obs_ss': bias_obs_ss, 'rmse_h1': rmse_h1, 'rmse_h3': rmse_h3,
        'e_h2_ctrl': e_h2_ctrl, 'e_obs': e_obs, 'mask_ss': mask_ss,
        'seg_info': seg_info,
    }

def print_summary(res):
    m = compute_metrics(res)
    avg_ms = np.mean(res['step_ms']); p99_ms = np.percentile(res['step_ms'], 99)
    print(f"\n{'='*62}")
    print(f"  RESUMEN — Koopman v4b-fix SIMULACION (RAMP={'ON' if ENABLE_REF_RAMP else 'OFF'})")
    print(f"{'='*62}")
    print(f"  RMSE ctrl h2 (SS):       {m['rmse_ctrl']:.4f} cm")
    print(f"  RMSE ctrl h2 (trans.):   {m['rmse_trans']:.4f} cm")
    print(f"  Bias ctrl h2 (SS):       {m['bias_ctrl']:.4f} cm")
    print(f"  RMSE obs h2 (SS):        {m['rmse_obs_ss']:.4f} cm")
    print(f"  RMSE est h1/h3 (SS):     {m['rmse_h1']:.4f} / {m['rmse_h3']:.4f} cm")
    print(f"  Tiempo medio/p99:        {avg_ms:.1f} / {p99_ms:.1f} ms")
    print(f"\n  Asentamiento +-5% por segmento (diagnostico de rampa):")
    for i, (r, ts5, os_) in enumerate(m['seg_info']):
        print(f"    Seg.{i+1} (ref={r*100:.0f}cm): Ts5%={ts5:.1f}s  OS={os_:.2f}cm")
    print(f"{'='*62}")

def plot_results(res, save_dir=output_dir):
    T   = res['T_hist']
    ref = np.append(res['yref_v'], res['yref_v'][-1])
    rct = res['ref_change_times_s']
    m   = compute_metrics(res)
    avg_ms = np.mean(res['step_ms']); p99_ms = np.percentile(res['step_ms'], 99)

    fig, axes = plt.subplots(3, 2, figsize=(13, 11))
    fig.suptitle(
        f'Koopman Closed-loop — Simulación | RAMP={"ON" if ENABLE_REF_RAMP else "OFF"} | '
        f'RMSE_SS ctrl={m["rmse_ctrl"]:.3f}cm | est h₂={m["rmse_obs_ss"]:.3f}cm | {avg_ms:.0f}ms/paso',
        fontsize=9, fontweight='bold')

    ax = axes[0, 0]
    ax.plot(T, res['X_true'][0]*100, 'b', lw=1.5, label='$h_1$ real')
    ax.plot(T, res['X_hat'][0]*100,  'r--', lw=1.5, label='$h_1$ est (Koopman)')
    ax.set_ylabel('$h_1$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tanque 1'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(T, res['X_true'][1]*100, 'b', lw=2, label='$h_2$ real')
    ax.plot(T, res['X_hat'][1]*100,  'r--', lw=2, label='$h_2$ est (Koopman)')
    ax.stairs(ref[:-1]*100, T, color='k', ls='--', lw=1.5, label='Referencia')
    ax.set_ylabel('$h_2$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tanque 2 — NO medido'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    ax.plot(T, res['X_true'][2]*100, 'b', lw=1.5, label='$h_3$ real')
    ax.plot(T, res['X_hat'][2]*100,  'r--', lw=1.5, label='$h_3$ est (Koopman)')
    ax.set_ylabel('$h_3$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tanque 3'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    bias_ctrl = float(np.mean(m['e_h2_ctrl'][m['mask_ss']]))
    ax = axes[1, 1]
    ax.plot(T, m['e_h2_ctrl'], 'r', lw=1.5,
            label=f'RMSE_SS={m["rmse_ctrl"]:.3f}cm | Bias={bias_ctrl:.3f}cm')
    ax.axhline(0, color='k', ls=':')
    ax.axvline(400, color='gray', lw=0.8, ls=':', label='t=400s')
    for i, tc in enumerate(rct):
        ax.axvspan(tc, tc + SETTLING_S, alpha=0.12, color='orange',
                   label='transitorio excluido' if i == 0 else None)
    ax.set_ylabel('Error ctrl $h_2$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Error de seguimiento'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[2, 0]
    ax.stairs(res['U_hist'][0], T, color='b', lw=1.5, label='$u_1$ (Bomba 1)')
    ax.stairs(res['U_hist'][1], T, color='r', lw=1.5, label='$u_3$ (Bomba 2)')
    ax.set_ylim([-0.05, 1.1]); ax.set_ylabel('Control [0–1]'); ax.set_xlabel('t [s]')
    ax.set_title('Señales de control'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[2, 1]
    ax.plot(res['step_ms'], 'k', lw=0.8, alpha=0.7)
    ax.axhline(avg_ms, color='b', lw=1.5, ls='--', label=f'Media={avg_ms:.1f} ms')
    ax.axhline(p99_ms, color='r', lw=1.5, ls=':',  label=f'P99={p99_ms:.1f} ms')
    ax.set_ylabel('Tiempo [ms]'); ax.set_xlabel('Paso k')
    ax.set_title('Tiempo de cómputo por paso'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    tag = 'ramp_on' if ENABLE_REF_RAMP else 'ramp_off'
    fname = save_dir / f'koopman_v4b_fix_sim_{tag}.png'
    plt.savefig(fname, dpi=180, bbox_inches='tight'); plt.close()
    print(f"\n  Figura guardada: {fname}")

# ============================================================================
# MAIN
# ============================================================================
def main():
    print("="*65)
    print("  Koopman Closed-Loop v4b-fix — SIMULACION")
    print(f"  RAMP={'ON' if ENABLE_REF_RAMP else 'OFF'} | KI={KI_WEIGHT} | "
          f"Z_INT_LIMIT=+-{Z_INT_LIMIT*100:.1f}cm.s")
    print("="*65)

    SS_NOM = {}
    for h2r in [0.10, 0.15, 0.20]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is not None:
            SS_NOM[h2r] = (x_ss, u_ss)
            print(f"  h2={h2r:.2f}m -> x_ss={np.round(x_ss,4)} | u_ss={np.round(u_ss,4)}")

    A_d, B_d, nz, nu = train_edmd(verbose=True)
    L_d, C_obs = design_observer(A_d, nz)
    mpc_args = build_mpc_with_integrator(A_d, B_d, nz, nu)
    solver, N_mpc, umin, umax, lbw, ubw, lbg, ubg, nz_aug, nu2 = mpc_args

    res = run_simulation(A_d, B_d, L_d, C_obs,
                         solver, N_mpc, umin, umax, lbw, ubw, lbg, ubg,
                         nz_aug, nu2, nz, SS_NOM)

    print_summary(res)
    plot_results(res)

if __name__ == '__main__':
    main()

  Koopman Closed-Loop v4b-fix — SIMULACION
  RAMP=ON | KI=80.0 | Z_INT_LIMIT=+-1.5cm.s
  h2=0.10m -> x_ss=[0.1139 0.1    0.0818] | u_ss=[0.3283 0.0012]
  h2=0.15m -> x_ss=[0.1675 0.15   0.1269] | u_ss=[0.4029 0.0012]
  h2=0.20m -> x_ss=[0.2207 0.2    0.1727] | u_ss=[0.4659 0.0012]

Entrenando EDMD (parametros planta real, v4b-fix)...
  8220 muestras de entrenamiento
  RMSE one-step: 9.74e-05
  |lambda| in [0.03444, 0.99990]

Disenando ganancia L (DARE con Q=0.1*I)...
  q=0.1: max|lambda(A-LC)|=0.95481, max|L|=0.2440
  tau_decay = 43.3 s

Construyendo KoopmanMPC v4b-fix-sim con integrador (ki=80.0)...
  KoopmanMPC v4b-fix-sim listo (nz_aug=9)
  Peso terminal: wk=5 en ultimos 3 pasos

  SIMULACION Koopman CL v4b-fix | 0.10->0.20->0.15m | sigma=1mm
  KI_WEIGHT=80.0 | Z_INT_LIMIT=+-1.5cm.s | DEADBAND=3cm | RAMP=ON
       0 | h2_real=  9.939 | h2_est=  9.939 | ref=  10.0 | u1=0.452 |    27ms
     100 | h2_real= 10.048 | h2_est= 10.022 | ref=  10.0 | u1=0.273 |     8ms
     200 | h2_real=  9